# 02 · Classificador del tipus d'avaria

El conjunt agregat no té el tipus d'avaria; MobiFix sí, però només fins al 2020. Entrenem un classificador de text amb MobiFix i l'apliquem a la resta (apartat 4, pas 5, del disseny).

- Textos en molts idiomes: n-grames de caràcters (3 a 5) i regressió logística, sense traduir.
- **Criteri del disseny:** si la F1 macro és inferior a 0,6, els models es fan només amb MobiFix.
- La regularització (`C`) i el llindar de confiança es trien amb validació creuada dins de MobiFix: el llindar és el més baix amb el qual l'encert dels registres que es queden arriba al 90 %. Per sota, el registre queda com a "Desconegut".
- Els registres que ja són a MobiFix conserven l'etiqueta humana; només es prediu la resta.

> **Ús en l'entrada sobre la bateria:** B1 no fa servir el classificador. B2 i B3 fan servir les etiquetes humanes de MobiFix com a resultat principal, i el conjunt classificat només com a complement. El classificador **no està validat** fora de MobiFix (textos posteriors al 2020 i en altres idiomes), i així s'ha de dir quan se'n citin xifres.

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, cohen_kappa_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline

import comu

LLINDAR_F1 = 0.6
ENCERT_MINIM = 0.9
C = 10  # es comprova a la validació creuada de més avall

def classificador(c=C):
    return make_pipeline(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True),
        LogisticRegression(C=c, class_weight="balanced", max_iter=3000),
    )

## Dades d'entrenament

MobiFix amb descripció del problema i un tipus d'avaria conegut.

In [2]:
mobifix = comu.carrega_mobifix()
entrenament = mobifix[mobifix["problem"].notna() & (mobifix["grup_avaria"] != "Desconegut")]
print(f"Registres d'entrenament: {len(entrenament):,}")
entrenament["grup_avaria"].value_counts()

Registres d'entrenament: 1,540


grup_avaria
Pantalla                     630
Bateria i alimentació        243
Programari i configuració    215
Altres                       200
Botons i connectors          121
Port de càrrega               88
Danys interns i aigua         43
Name: count, dtype: int64

## Validació amb un 20 % reservat

In [3]:
X_tr, X_te, y_tr, y_te = train_test_split(
    entrenament["problem"], entrenament["grup_avaria"],
    test_size=0.2, stratify=entrenament["grup_avaria"], random_state=42)
model = classificador().fit(X_tr, y_tr)
prediccio = model.predict(X_te)
print(classification_report(y_te, prediccio, zero_division=0))

                           precision    recall  f1-score   support

                   Altres       0.77      0.60      0.68        40
    Bateria i alimentació       0.95      0.85      0.90        48
      Botons i connectors       0.84      0.88      0.86        24
    Danys interns i aigua       0.56      0.56      0.56         9
                 Pantalla       0.94      0.98      0.96       126
          Port de càrrega       0.80      0.67      0.73        18
Programari i configuració       0.72      0.88      0.79        43

                 accuracy                           0.86       308
                macro avg       0.80      0.77      0.78       308
             weighted avg       0.86      0.86      0.86       308



In [4]:
# Matriu de confusió en proporció de cada classe real (files)
pd.crosstab(y_te, prediccio, normalize="index").round(2)

col_0,Altres,Bateria i alimentació,Botons i connectors,Danys interns i aigua,Pantalla,Port de càrrega,Programari i configuració
grup_avaria,,,,,,,
Altres,0.60,0.00,0.02,0.05,0.10,0.00,0.22
Bateria i alimentació,0.00,0.85,0.00,0.00,0.06,0.06,0.02
Botons i connectors,0.04,0.00,0.88,0.00,0.00,0.00,0.08
Danys interns i aigua,0.11,0.00,0.33,0.56,0.00,0.00,0.00
Pantalla,0.01,0.00,0.00,0.00,0.98,0.00,0.01
Port de càrrega,0.06,0.06,0.00,0.11,0.00,0.67,0.11
Programari i configuració,0.07,0.02,0.00,0.00,0.02,0.00,0.88


## Validació creuada i decisió

In [5]:
cv = StratifiedKFold(5, shuffle=True, random_state=42)
for c in (1, 10, 30):
    f1_c = cross_val_score(classificador(c), entrenament["problem"], entrenament["grup_avaria"],
                           cv=cv, scoring="f1_macro")
    print(f"C = {c:>2}: F1 macro {f1_c.mean():.3f} ± {f1_c.std():.3f}")
f1 = cross_val_score(classificador(), entrenament["problem"], entrenament["grup_avaria"],
                     cv=cv, scoring="f1_macro")
print(f"\nModel triat (C = {C}): F1 macro {f1.mean():.2f} ± {f1.std():.2f}")
USA_CLASSIFICADOR = f1.mean() >= LLINDAR_F1
print("Decisió:", "es fa servir el classificador al conjunt agregat" if USA_CLASSIFICADOR
      else "els models es fan només amb MobiFix")

C =  1: F1 macro 0.734 ± 0.016


C = 10: F1 macro 0.758 ± 0.027


C = 30: F1 macro 0.753 ± 0.025



Model triat (C = 10): F1 macro 0.76 ± 0.03
Decisió: es fa servir el classificador al conjunt agregat


## Llindar de confiança

Encert dels registres que es queden segons el llindar de probabilitat, amb prediccions de validació creuada.

In [6]:
prob_cv = cross_val_predict(classificador(), entrenament["problem"], entrenament["grup_avaria"],
                            cv=cv, method="predict_proba")
classes = np.sort(entrenament["grup_avaria"].unique())
encert = classes[prob_cv.argmax(axis=1)] == entrenament["grup_avaria"].to_numpy()
confianca = prob_cv.max(axis=1)
llindars = pd.DataFrame([{"llindar": t, "es_queden": (confianca >= t).mean(),
                          "encert": encert[confianca >= t].mean()} for t in np.arange(0, 0.75, 0.05)])
display(llindars.round(3))
LLINDAR_CONFIANCA = llindars.loc[llindars["encert"] >= ENCERT_MINIM, "llindar"].min()
print(f"Llindar triat: {LLINDAR_CONFIANCA:.2f}")

,llindar,es_queden,encert
0,0.00,1.000,0.840
1,0.05,1.000,0.840
2,0.10,1.000,0.840
3,0.15,1.000,0.840
4,0.20,1.000,0.840
5,0.25,0.996,0.842
6,0.30,0.976,0.853
7,0.35,0.946,0.865
8,0.40,0.917,0.877
9,0.45,0.882,0.891


Llindar triat: 0.55


## Creuar amb MobiFix

Els registres de MobiFix també són al conjunt agregat. Els creuem per proveïdor, data i text normalitzat (sense puntuació), i en aquests casos fem servir l'etiqueta humana.

In [7]:
mobils = pd.read_csv(comu.PROCESSADES / "mobils.csv")

def clau(df):
    return df["data_provider"] + "|" + df["event_date"].astype(str) + "|" + df["problem"].map(comu.normalitza_text)

# Al conjunt net no hi ha event_date: el recuperem de les dades en brut
brut = comu.carrega_aggregate()[["id", "event_date"]]
mobils = mobils.merge(brut, on="id", how="left")

etiquetats = mobifix[mobifix["problem"].notna()].assign(clau=clau)
# Si la mateixa clau té etiquetes diferents, ens quedem la més freqüent
etiqueta_humana = etiquetats.groupby("clau")["grup_avaria"].agg(lambda s: s.mode().iloc[0])

mobils["clau"] = np.where(mobils["problem"].notna(), clau(mobils.fillna({"problem": ""})), "")
mobils["grup_avaria"] = mobils["clau"].map(etiqueta_humana)
mobils["font_avaria"] = np.where(mobils["grup_avaria"].notna(), "MobiFix", None)
print(f"Registres creuats amb MobiFix: {mobils['grup_avaria'].notna().sum():,} "
      f"(de {(mobils['any_event'] <= 2020).sum():,} fins al 2020)")

Registres creuats amb MobiFix: 1,407 (de 2,057 fins al 2020)


## Aplicar el classificador a la resta

In [8]:
model_final = classificador().fit(entrenament["problem"], entrenament["grup_avaria"])

per_predir = mobils["grup_avaria"].isna() & mobils["problem"].notna()
if USA_CLASSIFICADOR:
    prob = model_final.predict_proba(mobils.loc[per_predir, "problem"])
    mobils.loc[per_predir, "grup_avaria"] = model_final.classes_[prob.argmax(axis=1)]
    mobils.loc[per_predir, "prob_avaria"] = prob.max(axis=1)
    mobils.loc[per_predir, "font_avaria"] = "classificador"
    poc_segur = per_predir & (mobils["prob_avaria"] < LLINDAR_CONFIANCA)
    mobils.loc[poc_segur, "grup_avaria"] = "Desconegut"
    print(f"Predits: {per_predir.sum():,}  ·  amb confiança < {LLINDAR_CONFIANCA:.2f}: {poc_segur.sum():,}")
    print("L'encert real al conjunt agregat pot ser més baix: hi ha idiomes i anys que MobiFix no té.")

mobils["grup_avaria"] = mobils["grup_avaria"].fillna("Desconegut")
mobils["font_avaria"] = mobils["font_avaria"].fillna("sense tipus")
pd.crosstab(mobils["grup_avaria"], mobils["font_avaria"], margins=True)

Predits: 3,020  ·  amb confiança < 0.55: 1,378
L'encert real al conjunt agregat pot ser més baix: hi ha idiomes i anys que MobiFix no té.


font_avaria,MobiFix,classificador,sense tipus,All
grup_avaria,,,,
Altres,184,235,0,419
Bateria i alimentació,200,302,0,502
Botons i connectors,108,90,0,198
Danys interns i aigua,25,36,0,61
Desconegut,78,1378,1225,2681
Pantalla,584,381,0,965
Port de càrrega,77,168,0,245
Programari i configuració,151,430,0,581
All,1407,3020,1225,5652


In [9]:
mobils.drop(columns=["clau"]).to_csv(comu.PROCESSADES / "mobils_classificats.csv", index=False)
print(f"Desat: {len(mobils):,} registres")

Desat: 5,652 registres
